# Story-Unfolding Name Alignment — Qwen 2.5 7B
### How does the hidden state align with suspect name embeddings as evidence accumulates?

---
**Core idea:** Run a forward pass after each story event. At each checkpoint, take the final-token hidden state x⁽ᴸ⁾ from the **last layer** and measure its cosine similarity to the embedding vectors of each name (John, Michael, Sarah).

As alibis are established and Michael is placed at the scene, we should see:
- **Michael's** alignment rising
- **John's** and **Sarah's** alignment dropping after their alibis land

---
**Story events (x-axis checkpoints):**
1. Intro — three people in a house
2. Crime — crash, police arrive
3. John's alibi — visiting relatives
4. Sarah's alibi — conference, train records
5. Evidence — struggle, muddy footprints
6. Michael seen — returning through garden gate
7. Conclusion prompt — "the murderer was"

---
## ✅ Before running
1. `Runtime → Change runtime type → T4 GPU`
2. No HuggingFace token needed
---

In [2]:
# ── Cell 1: Install ───────────────────────────────────────────────────────────
!pip install -q transformers>=4.42.0 accelerate>=0.30.0 bitsandbytes>=0.43.0

import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

PyTorch : 2.10.0+cpu
CUDA    : False


In [ ]:
# ── Cell 2: Load Qwen 2.5 7B in 4-bit ────────────────────────────────────────
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = 'Qwen/Qwen2.5-7B'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
)

print('Loading tokenizer ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print('Loading model in 4-bit ... (~4 GB, ~3 min)')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    output_hidden_states=True,
)
model.eval()

n_layers = model.config.num_hidden_layers
d_model  = model.config.hidden_size
print(f'\n✅ Loaded  |  layers={n_layers}  d_model={d_model}')

Loading tokenizer ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading model in 4-bit ... (~4 GB, ~3 min)


The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

In [ ]:
# ── Cell 3: Define story checkpoints ─────────────────────────────────────────
# Each entry is a prefix of the story — a snapshot of what the model has seen so far.
# The forward pass runs on the full prefix; we read the final-token hidden state.

NAMES   = ['John', 'Michael', 'Sarah']
MURDERER = 'Michael'
PALETTE  = {'John': '#457b9d', 'Michael': '#e63946', 'Sarah': '#2a9d8f'}

CHECKPOINTS = [
    (
        'Intro\n(three suspects)',
        'There was an old townhouse at the end of a quiet street. '
        'Three people lived there: John, Michael, and Sarah. '
        'They had shared the house for years, though lately the atmosphere '
        'had become tense after several arguments about money that had gone missing.'
    ),
    (
        'Crime\n(crash, police)',
        'There was an old townhouse at the end of a quiet street. '
        'Three people lived there: John, Michael, and Sarah. '
        'They had shared the house for years, though lately the atmosphere '
        'had become tense after several arguments about money that had gone missing. '
        'One evening the neighbors heard a loud crash followed by silence. '
        'The police arrived later that night to find a terrible crime had taken place inside.'
    ),
    (
        "John's alibi\n(visiting relatives)",
        'There was an old townhouse at the end of a quiet street. '
        'Three people lived there: John, Michael, and Sarah. '
        'They had shared the house for years, though lately the atmosphere '
        'had become tense after several arguments about money that had gone missing. '
        'One evening the neighbors heard a loud crash followed by silence. '
        'The police arrived later that night to find a terrible crime had taken place inside. '
        'John had been visiting relatives in another town that entire day and '
        'multiple witnesses confirmed he never left.'
    ),
    (
        "Sarah's alibi\n(conference, train records)",
        'There was an old townhouse at the end of a quiet street. '
        'Three people lived there: John, Michael, and Sarah. '
        'They had shared the house for years, though lately the atmosphere '
        'had become tense after several arguments about money that had gone missing. '
        'One evening the neighbors heard a loud crash followed by silence. '
        'The police arrived later that night to find a terrible crime had taken place inside. '
        'John had been visiting relatives in another town that entire day and '
        'multiple witnesses confirmed he never left. '
        'Sarah was attending a conference several hours away and had '
        'train records and hotel receipts confirming her alibi beyond any doubt.'
    ),
    (
        'Physical evidence\n(footprints, struggle)',
        'There was an old townhouse at the end of a quiet street. '
        'Three people lived there: John, Michael, and Sarah. '
        'They had shared the house for years, though lately the atmosphere '
        'had become tense after several arguments about money that had gone missing. '
        'One evening the neighbors heard a loud crash followed by silence. '
        'The police arrived later that night to find a terrible crime had taken place inside. '
        'John had been visiting relatives in another town that entire day and '
        'multiple witnesses confirmed he never left. '
        'Sarah was attending a conference several hours away and had '
        'train records and hotel receipts confirming her alibi beyond any doubt. '
        'When the detectives examined the house, they noticed signs of a struggle in the study '
        'and found a set of muddy footprints leading from the garden door to the hallway.'
    ),
    (
        'Michael seen\n(garden gate)',
        'There was an old townhouse at the end of a quiet street. '
        'Three people lived there: John, Michael, and Sarah. '
        'They had shared the house for years, though lately the atmosphere '
        'had become tense after several arguments about money that had gone missing. '
        'One evening the neighbors heard a loud crash followed by silence. '
        'The police arrived later that night to find a terrible crime had taken place inside. '
        'John had been visiting relatives in another town that entire day and '
        'multiple witnesses confirmed he never left. '
        'Sarah was attending a conference several hours away and had '
        'train records and hotel receipts confirming her alibi beyond any doubt. '
        'When the detectives examined the house, they noticed signs of a struggle in the study '
        'and found a set of muddy footprints leading from the garden door to the hallway. '
        'Earlier that evening, one of the neighbors recalled seeing Michael '
        'returning through the garden gate shortly before the incident occurred.'
    ),
    (
        'Conclusion prompt\n("the murderer was")',
        'There was an old townhouse at the end of a quiet street. '
        'Three people lived there: John, Michael, and Sarah. '
        'They had shared the house for years, though lately the atmosphere '
        'had become tense after several arguments about money that had gone missing. '
        'One evening the neighbors heard a loud crash followed by silence. '
        'The police arrived later that night to find a terrible crime had taken place inside. '
        'John had been visiting relatives in another town that entire day and '
        'multiple witnesses confirmed he never left. '
        'Sarah was attending a conference several hours away and had '
        'train records and hotel receipts confirming her alibi beyond any doubt. '
        'When the detectives examined the house, they noticed signs of a struggle in the study '
        'and found a set of muddy footprints leading from the garden door to the hallway. '
        'Earlier that evening, one of the neighbors recalled seeing Michael '
        'returning through the garden gate shortly before the incident occurred. '
        'After carefully reviewing all the evidence, the investigators '
        'concluded that the murderer was'
    ),
]

labels = [cp[0] for cp in CHECKPOINTS]
texts  = [cp[1] for cp in CHECKPOINTS]
print(f'Defined {len(CHECKPOINTS)} story checkpoints')
for i, (lbl, txt) in enumerate(CHECKPOINTS):
    toks = tokenizer(txt, return_tensors='pt')['input_ids'].shape[1]
    print(f'  [{i+1}] {lbl.split(chr(10))[0]:<30}  {toks} tokens')

In [ ]:
# ── Cell 4: Get name embedding vectors ───────────────────────────────────────
import numpy as np

E = model.get_input_embeddings().weight.float().detach().cpu().numpy()
print(f'Embedding matrix E : {E.shape}')

def get_token_id(word):
    # Try space-prefixed first (word-initial position in Qwen tokenizer)
    for candidate in [' ' + word, word]:
        ids = tokenizer.encode(candidate, add_special_tokens=False)
        if ids:
            tid = ids[0]
            print(f"  '{word}' → token id {tid}  (decoded: '{tokenizer.decode([tid])}')")
            return tid

print('Name token IDs:')
name_ids  = {name: get_token_id(name) for name in NAMES}
name_embs = {name: E[name_ids[name]]  for name in NAMES}

# Print cosine similarity between name pairs — sanity check
print('\nCosine similarity between name embeddings (should be low — distinct concepts):')
for i, n1 in enumerate(NAMES):
    for n2 in NAMES[i+1:]:
        e1 = name_embs[n1] / (np.linalg.norm(name_embs[n1]) + 1e-8)
        e2 = name_embs[n2] / (np.linalg.norm(name_embs[n2]) + 1e-8)
        print(f'  sim({n1}, {n2}) = {float(e1 @ e2):.4f}')

In [ ]:
# ── Cell 5: Forward pass at each checkpoint ───────────────────────────────────
# For each story prefix:
#   - tokenize & run model
#   - extract final-token hidden state from the LAST layer  x^(L)
#   - compute cosine similarity to each name embedding
#   - also record what the model would predict next (top-5)

def cosine_sim_vec(vec_a, vec_b):
    a = vec_a / (np.linalg.norm(vec_a) + 1e-8)
    b = vec_b / (np.linalg.norm(vec_b) + 1e-8)
    return float(a @ b)

results = []   # list of dicts, one per checkpoint

for idx, (label, text) in enumerate(CHECKPOINTS):
    short_label = label.split('\n')[0]
    print(f'[{idx+1}/{len(CHECKPOINTS)}] {short_label} ...', end=' ')

    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    seq_len = inputs['input_ids'].shape[1]

    with torch.no_grad():
        out = model(**inputs)

    # Final-layer hidden state of the last token
    x_final = out.hidden_states[-1][0, -1, :].float().cpu().numpy()  # (d_model,)

    # Cosine similarity to each name
    sims = {name: cosine_sim_vec(x_final, name_embs[name]) for name in NAMES}

    # Cosine distance to each name
    dists = {name: 1.0 - sims[name] for name in NAMES}

    # Top-5 next token predictions
    logits = out.logits[0, -1, :].float()
    probs  = torch.softmax(logits, dim=-1)
    top5   = torch.topk(probs, 5)
    top5_preds = [
        (tokenizer.decode([tid]).strip(), float(p))
        for tid, p in zip(top5.indices.tolist(), top5.values.tolist())
    ]

    results.append({
        'label':      label,
        'seq_len':    seq_len,
        'x_final':    x_final,
        'sims':       sims,
        'dists':      dists,
        'top5':       top5_preds,
        'winner_sim': max(sims, key=sims.get),
    })
    winner = max(sims, key=sims.get)
    print(f'done  ({seq_len} tokens)  '
          f'John={sims["John"]:+.4f}  Michael={sims["Michael"]:+.4f}  Sarah={sims["Sarah"]:+.4f}  '
          f'→ leading: {winner}')

print('\n✅ All checkpoints processed')

In [ ]:
# ── Cell 6: Print full table ──────────────────────────────────────────────────
print(f'{"Checkpoint":<35} {"John":>8} {"Michael":>9} {"Sarah":>8} {"Leading":>10}  Top prediction')
print('─' * 100)
for r in results:
    lbl     = r['label'].replace('\n', ' ')
    top_tok = r['top5'][0][0]
    top_p   = r['top5'][0][1]
    correct = '✅' if r['winner_sim'] == MURDERER else '  '
    print(f"{lbl:<35} {r['sims']['John']:>8.4f} {r['sims']['Michael']:>9.4f} "
          f"{r['sims']['Sarah']:>8.4f} {r['winner_sim']:>10}  "
          f"{correct}  '{top_tok}' ({top_p:.3f})'")

print('\nTop-5 predictions at final checkpoint ("the murderer was ___"):')
for tok, prob in results[-1]['top5']:
    bar = '█' * int(prob * 600)
    tag = '  ← ✅' if tok.lower() == MURDERER.lower() else ''
    print(f'  {tok:<15} {prob:.4f}  {bar}{tag}')

In [ ]:
# ── Cell 7: Main plot — cosine similarity as story unfolds ────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe

x      = np.arange(len(results))
x_lbls = [r['label'] for r in results]

# Key event x positions for vertical annotations
EVENTS = {
    2: ("John's\nalibi",   '#457b9d'),
    3: ("Sarah's\nalibi",  '#2a9d8f'),
    5: ("Michael\nseen",   '#e63946'),
}

fig, ax = plt.subplots(figsize=(14, 6))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#161b22')

# ── Draw vertical event shading ───────────────────────────────────────────────
for xi, (evt_label, evt_color) in EVENTS.items():
    ax.axvline(xi, color=evt_color, linewidth=1.2, linestyle=':', alpha=0.6, zorder=1)
    ax.text(xi + 0.05, ax.get_ylim()[1] if ax.get_ylim()[1] != 0 else 0.05,
            evt_label, color=evt_color, fontsize=8.5,
            va='top', ha='left', style='italic')

# ── Draw similarity lines per name ───────────────────────────────────────────
LWIDTH  = {'John': 2.0, 'Michael': 3.0, 'Sarah': 2.0}
LSTYLE  = {'John': '--', 'Michael': '-', 'Sarah': '-.'}
ZORDER  = {'John': 3, 'Michael': 5, 'Sarah': 3}
MARKER  = {'John': 'o', 'Michael': '*', 'Sarah': 'o'}
MSIZE   = {'John': 7,   'Michael': 12,  'Sarah': 7}

for name in NAMES:
    ys = [r['sims'][name] for r in results]

    ax.plot(x, ys,
            color=PALETTE[name],
            linewidth=LWIDTH[name],
            linestyle=LSTYLE[name],
            marker=MARKER[name],
            markersize=MSIZE[name],
            zorder=ZORDER[name],
            label=name,
            path_effects=[pe.withStroke(linewidth=4, foreground='#0d1117')])

    # Annotate final value
    final_y = ys[-1]
    ax.annotate(
        f'{name}  {final_y:+.4f}',
        xy=(x[-1], final_y),
        xytext=(x[-1] + 0.15, final_y),
        color=PALETTE[name], fontsize=10, va='center',
        fontweight='bold' if name == MURDERER else 'normal')

# ── Re-draw vertical lines on top of paths ────────────────────────────────────
y_min = min(r['sims'][n] for r in results for n in NAMES)
y_max = max(r['sims'][n] for r in results for n in NAMES)
y_pad = (y_max - y_min) * 0.12

for xi, (evt_label, evt_color) in EVENTS.items():
    ax.axvline(xi, color=evt_color, linewidth=1.2, linestyle=':', alpha=0.5, zorder=2)
    ax.text(xi + 0.06, y_max + y_pad * 0.3,
            evt_label, color=evt_color, fontsize=8.5,
            va='bottom', ha='left', style='italic')

# ── Winner shading for final checkpoint ──────────────────────────────────────
ax.axvspan(5.5, len(results)-0.5, alpha=0.06, color=PALETTE[MURDERER])

# ── Zero line ─────────────────────────────────────────────────────────────────
ax.axhline(0, color='#444', linewidth=0.8, linestyle=':', zorder=1)

# ── Axes & labels ─────────────────────────────────────────────────────────────
ax.set_xticks(x)
ax.set_xticklabels(x_lbls, color='white', fontsize=9.5)
ax.set_xlim(-0.4, len(results) - 0.4)
ax.set_ylim(y_min - y_pad, y_max + y_pad * 2.5)
ax.set_ylabel('cos( x⁽ᴸ⁾,  e_name )', color='white', fontsize=13)
ax.set_title(
    'Suspect Alignment as Story Unfolds — Qwen 2.5 7B\n'
    'Final-layer hidden state cosine similarity to each name embedding',
    color='white', fontsize=14, pad=14)
ax.tick_params(colors='white')
for sp in ax.spines.values():
    sp.set_edgecolor('#333')
ax.grid(axis='y', color='#2a2a2a', linestyle='--', linewidth=0.6, zorder=0)

# Legend
legend_patches = [
    mpatches.Patch(color=PALETTE[n],
                   label=f'{n}{"  ← murderer" if n == MURDERER else ""}')
    for n in NAMES
]
ax.legend(handles=legend_patches, facecolor='#1e2530', edgecolor='#444',
          labelcolor='white', fontsize=11, loc='upper left')

plt.tight_layout()
plt.savefig('fig_story_alignment.png', dpi=160,
            bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print('✅ Saved fig_story_alignment.png')

In [ ]:
# ── Cell 8: Delta plot — change in alignment at each new clue ─────────────────
# Shows the *shift* (Δ cosine similarity) caused by each new story event.
# Positive = aligned more with that name. Negative = model moved away.

fig, ax = plt.subplots(figsize=(14, 5))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#161b22')

bar_width = 0.25
x_delta   = np.arange(1, len(results))   # deltas start at checkpoint 2
x_lbls_d  = [r['label'] for r in results[1:]]

offsets = {'John': -bar_width, 'Michael': 0, 'Sarah': bar_width}

for name in NAMES:
    ys  = [r['sims'][name] for r in results]
    dys = [ys[i] - ys[i-1] for i in range(1, len(ys))]
    bars = ax.bar(
        x_delta + offsets[name], dys,
        width=bar_width * 0.85,
        color=PALETTE[name],
        alpha=0.85,
        edgecolor='white', linewidth=0.4,
        label=name, zorder=3)

ax.axhline(0, color='#888', linewidth=0.9, linestyle='-', zorder=2)

for xi, (evt_label, evt_color) in EVENTS.items():
    ax.axvline(xi, color=evt_color, linewidth=1.0, linestyle=':', alpha=0.5)

ax.set_xticks(x_delta)
ax.set_xticklabels(x_lbls_d, color='white', fontsize=9)
ax.set_ylabel('Δ cos( x⁽ᴸ⁾,  e_name )', color='white', fontsize=12)
ax.set_title(
    'Alignment Shift at Each New Story Event — Qwen 2.5 7B\n'
    'Δ cosine similarity to each name embedding (positive = model leans toward that name)',
    color='white', fontsize=13, pad=12)
ax.tick_params(colors='white')
for sp in ax.spines.values():
    sp.set_edgecolor('#333')
ax.grid(axis='y', color='#2a2a2a', linestyle='--', linewidth=0.6, zorder=0)
ax.legend(facecolor='#1e2530', edgecolor='#444', labelcolor='white', fontsize=11)

plt.tight_layout()
plt.savefig('fig_delta_alignment.png', dpi=160,
            bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print('✅ Saved fig_delta_alignment.png')

In [ ]:
# ── Cell 9: Download all outputs ──────────────────────────────────────────────
from google.colab import files
for fname in ['fig_story_alignment.png', 'fig_delta_alignment.png']:
    files.download(fname)
    print(f'⬇  {fname}')